# Notebook 35 — Corrected V4.2 ANN Serving Benchmark

This notebook stops model tuning and evaluates whether FAISS HNSW can replace full-catalog exact matrix search without materially changing the fixed Corrected V4.2 policy. HNSW generates candidates; the original author/popularity score and point-in-time masks are then reapplied.

Validation holdout only. Locked Test is not loaded. Use an A100 or L4 GPU with High-RAM.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT=DRIVE_ROOT/'data/gold/adaptive/v1'
BASE_GOLD_ROOT=DRIVE_ROOT/'data/gold/two_tower/v1_colab'
PROTOCOL_DIR=ADAPTIVE_ROOT/'evaluation_protocol/v2'
WORK_ROOT=Path('/content/recsys_v4_ann_benchmark')
SOURCE_ROOT=WORK_ROOT/'runtime'; HF_SNAPSHOT_ROOT=WORK_ROOT/'hf_snapshot'
CORRECTED_ROOT=HF_SNAPSHOT_ROOT/'corrected_sequences/v2'
V4_DIR=WORK_ROOT/'artifacts/corrected_v4'; REPORT_DIR=WORK_ROOT/'reports/ann'
DRIVE_REPORT_DIR=DRIVE_ROOT/'artifacts/adaptive/v4_ann_benchmark_v1'
for p in (SOURCE_ROOT,V4_DIR,REPORT_DIR,DRIVE_REPORT_DIR): p.mkdir(parents=True,exist_ok=True)
print('work root:',WORK_ROOT)

In [ ]:
required={'torch':'torch','pandas':'pandas','pyarrow':'pyarrow','huggingface_hub':'huggingface_hub>=0.32','faiss':'faiss-cpu>=1.9'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch, faiss
assert torch.cuda.is_available(),'Select an A100 or L4 GPU runtime.'
free_gib=shutil.disk_usage('/content').free/1024**3
assert free_gib>=15,f'At least 15 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print('GPU:',torch.cuda.get_device_name(0),'FAISS:',faiss.__version__,f'disk={free_gib:.1f} GiB')

In [ ]:
required_paths=[ADAPTIVE_ROOT/'sequence_examples/train',ADAPTIVE_ROOT/'sequence_examples/validation',BASE_GOLD_ROOT/'item_features/static',BASE_GOLD_ROOT/'transforms/numeric_stats.json',BASE_GOLD_ROOT/'vocabularies/user_active_degree',BASE_GOLD_ROOT/'vocabularies/video_type',BASE_GOLD_ROOT/'vocabularies/upload_type',PROTOCOL_DIR/'queries.parquet',PROTOCOL_DIR/'manifest.json']
missing_paths=[str(p) for p in required_paths if not p.exists()]
assert not missing_paths,'Missing Drive inputs:\n'+'\n'.join(missing_paths)
manifest=json.loads((PROTOCOL_DIR/'manifest.json').read_text())
assert manifest['builder_revision']=='2.1-session-history'
print('Drive/A2 preflight: PASS',manifest['queries'],manifest['users'])

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
HF_REPO_ID='khoa1505/recsys-artifacts'
RUNTIME_FILE='runtime/corrected_v4_ann_runtime_v1.zip'
RUNTIME_SHA256='85847923221cc6fd6cf3200905b47d2954fd748da59f8c216d18f3f95e1da06a'
V4_FILE='v4_corrected/v1/best_model.pt'
def normalize_token(raw):
    if isinstance(raw,bytes): raw=raw.decode()
    if isinstance(raw,str):
        raw=raw.strip(); return raw if raw.startswith('hf_') else None
    if isinstance(raw,dict):
        for value in raw.values():
            token=normalize_token(value)
            if token:return token
    return None
try: HF_TOKEN=normalize_token(userdata.get('HF_TOKEN'))
except Exception: HF_TOKEN=None
HF_TOKEN=HF_TOKEN or normalize_token(getpass('Paste Hugging Face READ token: '))
assert HF_TOKEN,'HF_TOKEN is missing or invalid.'
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block:=stream.read(8*1024*1024): digest.update(block)
    return digest.hexdigest()
runtime_zip=Path(hf_hub_download(HF_REPO_ID,RUNTIME_FILE,repo_type='model',token=HF_TOKEN))
v4_cached=Path(hf_hub_download(HF_REPO_ID,V4_FILE,repo_type='model',token=HF_TOKEN))
assert sha256(runtime_zip)==RUNTIME_SHA256,'Runtime ZIP hash mismatch'
assert v4_cached.stat().st_size>100_000_000,'Corrected V4 checkpoint is unexpectedly small'
snapshot_download(HF_REPO_ID,repo_type='model',token=HF_TOKEN,local_dir=HF_SNAPSHOT_ROOT,allow_patterns=['corrected_sequences/v2/manifest.json','corrected_sequences/v2/sequences/split=validation/**'])
corrected_manifest=json.loads((CORRECTED_ROOT/'manifest.json').read_text())
assert corrected_manifest['quality']['passed'] is True
target=V4_DIR/'best_model.pt'; target.unlink(missing_ok=True); target.symlink_to(v4_cached.resolve())
print('verified runtime/checkpoint/corrected validation delta: PASS')

In [ ]:
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None; archive.extractall(SOURCE_ROOT)
BENCHMARK_SCRIPT=SOURCE_ROOT/'scripts/benchmark_corrected_v4_ann.py'
assert BENCHMARK_SCRIPT.exists(); compile(BENCHMARK_SCRIPT.read_text(),str(BENCHMARK_SCRIPT),'exec')
print('runtime extraction/compile: PASS')

In [ ]:
command=[sys.executable,str(BENCHMARK_SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--corrected-root',str(CORRECTED_ROOT),'--corrected-v4-dir',str(V4_DIR),'--protocol-dir',str(PROTOCOL_DIR),'--output-dir',str(REPORT_DIR),'--author-weight','0.2','--popularity-weight','0.02','--recency-tau','3.0','--top-k','1000','--search-k','5000','--hnsw-m','32','--ef-construction','80','--ef-search-values','5000,7500,10000','--max-holdout-queries','1000','--query-batch-size','8','--item-batch-size','8192','--minimum-overlap-recall','0.95','--maximum-hitrate-drop','0.001','--device','cuda']
log_path=Path('/content/corrected_v4_ann_benchmark.log')
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with log_path.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0:
    tail='\n'.join(log_path.read_text(errors='replace').splitlines()[-240:])
    raise RuntimeError(f'ANN benchmark failed with exit code {code}:\n{tail}')
REPORT_JSON=REPORT_DIR/'corrected_v4_ann_benchmark.json'
assert REPORT_JSON.exists(),REPORT_JSON
print('ANN benchmark: PASS')

In [ ]:
import pandas as pd
report=json.loads(REPORT_JSON.read_text())
display(pd.read_csv(REPORT_DIR/'ann_benchmark.csv'))
print(json.dumps(report['acceptance'],indent=2))
print('exact ms/query:',report['exact_ms_per_query'])
print('index MiB:',report['index']['bytes']/1024**2)

In [ ]:
small=WORK_ROOT/'small_report'; shutil.rmtree(small,ignore_errors=True); small.mkdir()
for name in ('corrected_v4_ann_benchmark.json','ann_benchmark.csv'):
    shutil.copy2(REPORT_DIR/name,small/name); shutil.copy2(REPORT_DIR/name,DRIVE_REPORT_DIR/name)
shutil.copy2(log_path,small/log_path.name); shutil.copy2(log_path,DRIVE_REPORT_DIR/log_path.name)
archive=Path(shutil.make_archive('/content/corrected_v4_ann_serving_report','zip',small))
print('Index remains on ephemeral Colab disk and is intentionally not copied to Drive.')
print('Drive report:',DRIVE_REPORT_DIR,'download:',archive)
files.download(str(archive))

## Decision

A configuration passes only when mean exact Top-1000 overlap is at least 95%, target HitRate drops by at most 0.1 percentage point, and every query still returns 1000 valid candidates. If none passes, increase candidate over-generation or use an author-aware multi-index; do not modify model weights or inspect locked Test.